# Pokémon Battle Prediction — FDS Kaggle Competition

This notebook uses **only** the official competition files, clones the project from a public GitHub repository, and creates the three documented predictions. All code, methodological notes, and comments are in English.


## 1. Clone the source repository

Replace the placeholder with the actual public GitHub repository URL. Enable Kaggle Internet access to use `git clone`. Source code is **not** copied into this notebook.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/lucamartone/fds-pokemon.git"
REPO_DIR = Path("/kaggle/working/fds-pokemon")

if "<username>" in REPO_URL:
    raise ValueError("Replace REPO_URL with your public GitHub repository URL.")

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))


## 2. Locate competition data and build features

The feature builder only uses the available battle observations, the supplied type chart, and species statistics already present in the official JSONL files.


In [ ]:
from pathlib import Path
from pokemon.main import build_all, cross_validate, make_submission

train_files = list(Path("/kaggle/input").rglob("train.jsonl"))
if len(train_files) != 1:
    raise FileNotFoundError(f"Expected exactly one train.jsonl, found {len(train_files)}")

DATA_DIR = train_files[0].parent
for filename in ("test.jsonl", "type_chart_gen2.csv"):
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {DATA_DIR}")

features = build_all(DATA_DIR)
print("Training features:", features["A"].shape)
print("Test features:", features["B"].shape)


## 3. Validation

We use stratified five-fold cross-validation with **nested target encoding**. In every fold, the validation labels are excluded from the target-encoding statistics. Expected reference results are 69.50% for LightGBM, 69.70% for the LGBM/XGB blend, and 69.79% for the three-model blend. These are training OOF estimates, not Kaggle test scores.


In [ ]:
scores = cross_validate(features)
for name, accuracy in scores.items():
    print(f"{name}: {accuracy:.2%}")


## 4. Submission 1 — LightGBM

A strongly regularized tree ensemble using team, resource, tactical, and cross-fitted target-encoding features. Five-fold out-of-fold accuracy: **69.50%**.


In [ ]:
submission_1 = make_submission(features, "sub1_lgbm", "/kaggle/working")
print(submission_1)


## 5. Submission 2 — LightGBM + XGBoost

An **equal 50/50 average** of the two probability predictions. XGBoost provides a different boosting implementation and regularization regime. Five-fold out-of-fold accuracy: **69.70%**.


In [ ]:
submission_2 = make_submission(features, "sub2_lgb_xgb", "/kaggle/working")
print(submission_2)


## 6. Submission 3 — LightGBM + XGBoost + CatBoost

A **45% / 45% / 10% probability blend**, selected using labeled-training cross-validation. Five-fold out-of-fold accuracy: **69.79%**, the highest observed result, but far below the unverified 80% goal.


In [ ]:
submission_3 = make_submission(features, "sub3_lgb_xgb_cat", "/kaggle/working")
print(submission_3)


## 7. Validate submission format

All predictions must have the same `battle_id` values and order as the official competition template, with `player_won` restricted to 0 or 1.


In [ ]:
import pandas as pd

sample_files = list(Path("/kaggle/input").rglob("sample_submission.csv"))
if len(sample_files) != 1:
    raise FileNotFoundError("Expected exactly one sample_submission.csv")

sample = pd.read_csv(sample_files[0])
for output in (submission_1, submission_2, submission_3):
    submission = pd.read_csv(output)
    assert submission.columns.tolist() == sample.columns.tolist()
    assert submission["battle_id"].equals(sample["battle_id"])
    assert submission["player_won"].isin([0, 1]).all()
    print(f"Validated {Path(output).name}: {len(submission)} predictions")


## 8. Additional research (not selected for submission)

A separate, documented experiment trained classical auxiliary regressors on observed health transitions and on ten-turn changes in team resources. The resulting XGBoost classifier reached **69.38%** five-fold out-of-fold accuracy, compared with **69.79%** for the selected ensemble. Therefore, the three submissions above remain unchanged. The code and methodology are available in `pokemon/transitions.py`, `pokemon/forecast.py`, `transition_experiment.py` and `docs/TRANSITION_EXPERIMENT.md` in the cloned repository.
